In [1]:
import os 
import numpy as np
import pandas as pd
import stark as sk
import anndata as ad
from tqdm import tqdm

In [5]:
path1 = '/Users/ckw/warehouse/metacell/stark/supercell'
result = pd.DataFrame(
        columns=[
            "mean_purity",
            "acc",
            "global_score",
            "wcos",
            "hwis",
            "compactness",
            "separation",
        ]
    )
pbar = tqdm(range(11,40), desc="MetaCell 数量", unit="num")
for num in pbar:
    file = path1+ f'/metacell_membership_{num}.csv'
    file = pd.read_csv(file)

    lb = []
    path = '/Users/ckw/warehouse/metacell/data/test_700_snm3c'
    for val in os.listdir(path):
        if val.endswith('.pairs'):
            lb.append(val.split('.pairs')[0].split('_')[1])
    lb = ['ExcNeuron' if x in ['L23', 'L4', 'L5', 'L6'] else x for x in lb]
    pca_vec = np.load('/Users/ckw/warehouse/metacell/stark/test_output/pca_vec_500000.npy')
    umap_vec = np.load('/Users/ckw/warehouse/metacell/stark/test_output/umap_vec_500000.npy')
    cell_embeddings = pca_vec  # 或者 umap_vec，取决于你想用哪个作为输入
    print(pca_vec.shape, umap_vec.shape, len(lb))
    adata = ad.AnnData(cell_embeddings)
    adata.obs['cell_type'] = lb
    adata.obsm['X_pca'] = pca_vec
    adata.obsm['X_umap'] = umap_vec
    adata.uns['X_pca'] = pca_vec    
    adata.uns['X_umap'] = umap_vec  
    adata.obs['metacell'] =np.array( file.iloc[:,1])
    adata.obs.columns = ['label', 'metacell']
    adata.obs['cell_id'] = np.array(file.iloc[:, 0])
    hdata = sk.create_hdata_from_adata(adata,
                                    data_dir="/Users/ckw/warehouse/metacell/data/test_700_snm3c",
                                output_dir="/Users/ckw/warehouse/metacell/stark/test_output",
                                genome_reference_path="/Users/ckw/warehouse/metacell/hg19.fa.chrom.sizes",
                                chrom_list=[f"chr{i}" for i in range(1, 23)],
                                resolution=[500000])
    purity_df, metrics = sk.tl.evaluate(hdata, hdata.obs['label'])

    vals = np.array(metrics.values())
    pres_df, metrics_summary = sk.tl.evaluate_metacell(
        hdata=hdata,
        use_view=None,  # 默认选第0个组学视角，也可以传入字典里的特定 key
        metric="euclidean",  # 或 'cosine'
    )

  
    print(vals)
    result.loc[num] = vals

result.to_csv('./benchmark/supercell.csv')

MetaCell 数量:   0%|          | 0/29 [00:00<?, ?num/s]/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)
/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)
MetaCell 数量:   7%|▋         | 2/29 [00:00<00:01, 13.83num/s]/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64

(700, 114) (700, 2) 700

正在计算评估指标...
✅ 指标计算完成！(发现 11 种细胞类型)
----------------------------------------
简单平均纯度 (Mean Purity)  : 0.7326
模型准确率 (Accuracy)      : 0.7343
全局加权分 (Global Score)  : 0.6208
过度融合指标 (WCOS)       : 0.9371
Hub 权重不纯度 (HWIS)     : 0.9752
----------------------------------------
✅ 评估指标计算完成，纯度得分(EP_v2等)已同步至 hdata.metacells。

正在计算 Metacell 空间结构指标 (Compactness & Separation)...
使用的特征视图: 500000 (距离度量: euclidean)
----------------------------------------
全局平均紧凑度得分 (Compactness Score): 0.5669 (值域 0~1，1=最紧凑/最佳)
全局平均分离度得分 (Separation Score) : 0.4235 (值域 0~1，1=最分离/最佳)
----------------------------------------
✅ 空间分布评估指标计算完成，得已并入 hdata.metacells 及 hdata.uns 缓存中。
dict_values([0.7325707088338645, 0.7342857142857143, 0.6207500708911138, 0.9371428571428572, 0.9751714285714286, 0.5669214381501093, 0.4234996114819451])
(700, 114) (700, 2) 700

正在计算评估指标...
✅ 指标计算完成！(发现 11 种细胞类型)
----------------------------------------
简单平均纯度 (Mean Purity)  : 0.7783
模型准确率 (Accuracy)      : 0.7686
全局加权分 (Glob

/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)
MetaCell 数量:  14%|█▍        | 4/29 [00:00<00:01, 13.33num/s]/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)
/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion wi

----------------------------------------
全局平均紧凑度得分 (Compactness Score): 0.5635 (值域 0~1，1=最紧凑/最佳)
全局平均分离度得分 (Separation Score) : 0.5656 (值域 0~1，1=最分离/最佳)
----------------------------------------
✅ 空间分布评估指标计算完成，得已并入 hdata.metacells 及 hdata.uns 缓存中。
dict_values([0.7931123082745947, 0.7685714285714286, 0.6443460462403483, 0.9371428571428572, 0.9791448979591837, 0.5634963272740936, 0.5656153806955594])
(700, 114) (700, 2) 700

正在计算评估指标...
✅ 指标计算完成！(发现 11 种细胞类型)
----------------------------------------
简单平均纯度 (Mean Purity)  : 0.8072
模型准确率 (Accuracy)      : 0.7686
全局加权分 (Global Score)  : 0.6336
过度融合指标 (WCOS)       : 0.9371
Hub 权重不纯度 (HWIS)     : 0.9792
----------------------------------------
✅ 评估指标计算完成，纯度得分(EP_v2等)已同步至 hdata.metacells。

正在计算 Metacell 空间结构指标 (Compactness & Separation)...
使用的特征视图: 500000 (距离度量: euclidean)
----------------------------------------
全局平均紧凑度得分 (Compactness Score): 0.5504 (值域 0~1，1=最紧凑/最佳)
全局平均分离度得分 (Separation Score) : 0.3273 (值域 0~1，1=最分离/最佳)
---------------------

MetaCell 数量:  21%|██        | 6/29 [00:00<00:01, 13.82num/s]/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)
/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)


...

/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)
/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)
MetaCell 数量:  69%|██████▉   | 20/29 [00:01<00:00, 12.74num/s]/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion w

✅ 指标计算完成！(发现 11 种细胞类型)
----------------------------------------
简单平均纯度 (Mean Purity)  : 0.8365
模型准确率 (Accuracy)      : 0.8157
全局加权分 (Global Score)  : 0.6164
过度融合指标 (WCOS)       : 0.9571
Hub 权重不纯度 (HWIS)     : 0.9913
----------------------------------------
✅ 评估指标计算完成，纯度得分(EP_v2等)已同步至 hdata.metacells。

正在计算 Metacell 空间结构指标 (Compactness & Separation)...
使用的特征视图: 500000 (距离度量: euclidean)
----------------------------------------
全局平均紧凑度得分 (Compactness Score): 0.5786 (值域 0~1，1=最紧凑/最佳)
全局平均分离度得分 (Separation Score) : 0.4164 (值域 0~1，1=最分离/最佳)
----------------------------------------
✅ 空间分布评估指标计算完成，得已并入 hdata.metacells 及 hdata.uns 缓存中。
dict_values([0.8364919127067929, 0.8157142857142857, 0.6163864513954442, 0.9571428571428572, 0.9913020408163266, 0.5786152160304049, 0.4164397519476871])
(700, 114) (700, 2) 700

正在计算评估指标...
✅ 指标计算完成！(发现 11 种细胞类型)
----------------------------------------
简单平均纯度 (Mean Purity)  : 0.8418
模型准确率 (Accuracy)      : 0.8157
全局加权分 (Global Score)  : 0.6094
过度融合指标 (WCOS)    

/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)
/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(cell_embeddings)
MetaCell 数量:  83%|████████▎ | 24/29 [00:01<00:00, 14.55num/s]/var/folders/vj/gx99370d7_z4fq339y_07z700000gn/T/ipykernel_86051/2540893898.py:28: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion w

✅ 指标计算完成！(发现 11 种细胞类型)
----------------------------------------
简单平均纯度 (Mean Purity)  : 0.8354
模型准确率 (Accuracy)      : 0.8157
全局加权分 (Global Score)  : 0.6125
过度融合指标 (WCOS)       : 0.9729
Hub 权重不纯度 (HWIS)     : 0.9930
----------------------------------------
✅ 评估指标计算完成，纯度得分(EP_v2等)已同步至 hdata.metacells。

正在计算 Metacell 空间结构指标 (Compactness & Separation)...
使用的特征视图: 500000 (距离度量: euclidean)
----------------------------------------
全局平均紧凑度得分 (Compactness Score): 0.5930 (值域 0~1，1=最紧凑/最佳)
全局平均分离度得分 (Separation Score) : 0.3760 (值域 0~1，1=最分离/最佳)
----------------------------------------
✅ 空间分布评估指标计算完成，得已并入 hdata.metacells 及 hdata.uns 缓存中。
dict_values([0.8354471933698117, 0.8157142857142857, 0.6125043041040884, 0.9728571428571429, 0.9929755102040816, 0.5930296280164178, 0.37602481187101])
(700, 114) (700, 2) 700

正在计算评估指标...
✅ 指标计算完成！(发现 11 种细胞类型)
----------------------------------------
简单平均纯度 (Mean Purity)  : 0.8334
模型准确率 (Accuracy)      : 0.8157
全局加权分 (Global Score)  : 0.6083
过度融合指标 (WCOS)      

MetaCell 数量: 100%|██████████| 29/29 [00:02<00:00, 13.80num/s]
